[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sterolose/sirius-statistics-2026/blob/main/notebooks/9_regression.ipynb)

## Семинар 9. Регрессионный анализ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan

Приготовим пингвинов как раньше.

In [ ]:
df = sns.load_dataset("penguins")
df.insert(2, "body_mass_kg", df.pop("body_mass_g") / 1000)
df = df.dropna()
adelie = df.loc[df["species"] == "Adelie"]
adelie.head(2)

Во всех заданиях используем уровень значимости $\alpha = 0.05$, для коэффициентов — двусторонние альтернативы.

### Задание 1

Построим простую линейную регрессию длины ласты от массы тела для Adelie. Сначала посмотрим на связь с помощью `sns.regplot`.

In [ ]:
sns.regplot(
    adelie, x="body_mass_kg", y="flipper_length_mm",
    scatter_kws={"s": 20, "alpha": 0.6}, ci=None,
)
plt.show()

Используем знакомую функцию [`ols`](https://www.statsmodels.org/stable/example_formulas.html). Свободный член добавляется в модель автоматически.

In [ ]:
model = ols("flipper_length_mm ~ body_mass_kg", adelie).fit()
model.summary()

В `summary` много информации. Выведем отдельно оценки коэффициентов, стандартные ошибки, p-value для $H_0: \beta_j = 0$ и 95%-е доверительные интервалы.

In [ ]:
pd.DataFrame({
    "coef": model.params,
    "SE": model.bse,
    "p-value": model.pvalues,
}).join(model.conf_int().rename(columns={0: "CI low", 1: "CI high"}))

In [ ]:
pd.Series({
    "R2": model.rsquared,
    "R2 adj": model.rsquared_adj,
    "RMSE": np.sqrt(np.mean(model.resid ** 2)),
    "F p-value": model.f_pvalue,
}).to_frame("value")

Главное — величина и смысл коэффициентов, их интервалы и диагностика остатков. $R^2$ показывает долю объясненной вариации, скорректированный $R^2$ учитывает число параметров, а RMSE здесь описывает ошибки на той же выборке. Общий F-тест проверяет, равны ли нулю все коэффициенты, кроме свободного члена.

Нормальность предполагается для ошибок модели, а не для исходных признаков. Проверим ее по остаткам. Гомоскедастичность означает постоянство дисперсии ошибок.

Используем тест Шапиро — Уилка и тест Бройша — Пагана (по умолчанию — вариант Кёнкера). Для второго берем последнее из четырех возвращаемых значений: p-value F-теста. В `model.model.exog` находится матрица признаков модели, включая свободный член.

In [ ]:
pd.Series({
    "Shapiro-Wilk": stats.shapiro(model.resid).pvalue,
    "Breusch-Pagan": het_breuschpagan(model.resid, model.model.exog)[3],
}, name="p-value").to_frame()

Постройте гистограмму и Q–Q plot остатков.

In [ ]:
# Your code here

Посмотрим на предсказанные значения $\hat y$ в зависимости от наблюдаемого $y$ и остатки $e = y - \hat y$ в зависимости от $\hat y$. Их можно получить как `model.fittedvalues` и `model.resid`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].scatter(adelie["flipper_length_mm"], model.fittedvalues, s=20, alpha=0.6)
axes[0].plot(axes[0].get_xlim(), axes[0].get_xlim(), "k--", linewidth=1, zorder=0)
axes[0].set(xlabel="Observed flipper length, mm", ylabel="Fitted flipper length, mm")

axes[1].scatter(model.fittedvalues, model.resid, s=20, alpha=0.6)
axes[1].axhline(0, color="k", linestyle="--", linewidth=1, zorder=0)
axes[1].set(xlabel="Fitted flipper length, mm", ylabel="Residual, mm")

plt.tight_layout()
plt.show()

### Задание 2

Теперь для тех же Adelie будем моделировать **массу тела** по длине ласты, размерам клюва и полу. `C(sex)` обозначает категориальный признак; строковые столбцы распознаются как категориальные и без `C`.

Запись `flipper_length_mm * C(sex)` включает оба основных эффекта и их взаимодействие. Она позволяет наклону зависимости массы от длины ласты различаться между полами.

In [ ]:
model = ols(
    "body_mass_kg ~ flipper_length_mm * C(sex) + bill_length_mm + bill_depth_mm",
    adelie,
).fit()

pd.DataFrame({
    "coef": model.params,
    "p-value": model.pvalues,
})

Упростим модель, начав с взаимодействия. Пока оно присутствует, сохраняем оба входящих в него основных эффекта.

В `statsmodels` сравнивать вложенные модели через `compare_f_test`.

In [ ]:
reduced = ols(
    "body_mass_kg ~ flipper_length_mm + C(sex) + bill_length_mm + bill_depth_mm",
    adelie,
).fit()

pd.Series(
    model.compare_f_test(reduced),
    index=["F", "p-value", "df difference"],
).to_frame("value")

Отказ от взаимодействия не дает статистически значимого ухудшения. Продолжим с сокращенной моделью.

In [ ]:
model = reduced
pd.DataFrame({"coef": model.params, "p-value": model.pvalues})

Уберите следующий незначимый признак и повторите сравнение вложенных моделей. Для итоговой модели выведите коэффициенты с интервалами, скорректированный $R^2$ и проверьте остатки. Постройте диаграммы рассеяния $\hat y$ от $y$ и $e$ от $\hat y$.

In [ ]:
# Your code here

### Задание 3

Перейдем к датасету [Trees](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/trees.html): измерения **31 срубленного дерева черемухи поздней** (*Prunus serotina*).

- `Girth` — диаметр ствола в дюймах, измеренный на высоте 4 фута 6 дюймов;
- `Height` — высота дерева в футах;
- `Volume` — объем древесины в кубических футах.

Будем моделировать объем по диаметру и высоте.

In [ ]:
trees = pd.read_csv(
    "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/trees.csv",
    index_col=0,
)
trees.head(2)

Посмотрим на обе зависимости по отдельности.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

for column, ax in zip(["Girth", "Height"], axes):
    sns.regplot(trees, x=column, y="Volume", ci=None, ax=ax)

plt.tight_layout()
plt.show()

Начнем с суммы двух признаков в формуле: это два отдельных коэффициента, а не один признак `Girth + Height`.

In [ ]:
model = ols("Volume ~ Girth + Height", trees).fit()

pd.DataFrame({"coef": model.params, "p-value": model.pvalues})

Выведите скорректированный $R^2$. Постройте график остатков от предсказанных значений и проверьте предпосылки модели. Обратите внимание на форму облака остатков даже при незначимом результате теста.

In [ ]:
# Your code here

Если приближать ствол цилиндром или конусом, объем пропорционален квадрату диаметра и высоте: $V \propto D^2 H$. Для начала заменим диаметр его квадратом.

В формуле `I(Girth ** 2)` означает обычную арифметическую операцию. Модель остается линейной по коэффициентам.

In [ ]:
model = ols("Volume ~ I(Girth ** 2) + Height", trees).fit()

pd.DataFrame({"coef": model.params, "p-value": model.pvalues})

Сравните скорректированный $R^2$ и график остатков с предыдущей моделью. Повторите тесты на предпосылки.

In [ ]:
# Your code here

Теперь используем произведение $D^2 H$. Двоеточие включает только произведение; в отличие от `*`, отдельные слагаемые автоматически не добавляются. Это отдельная модель, мотивированная геометрией ствола.

In [ ]:
model = ols("Volume ~ I(Girth ** 2):Height", trees).fit()

pd.DataFrame({"coef": model.params, "p-value": model.pvalues})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].scatter(trees["Volume"], model.fittedvalues)
axes[0].plot(axes[0].get_xlim(), axes[0].get_xlim(), "k--", linewidth=1, zorder=0)
axes[0].set(xlabel=r"Observed volume, ft$^3$", ylabel=r"Fitted volume, ft$^3$")

axes[1].scatter(model.fittedvalues, model.resid)
axes[1].axhline(0, color="k", linestyle="--", linewidth=1, zorder=0)
axes[1].set(xlabel=r"Fitted volume, ft$^3$", ylabel=r"Residual, ft$^3$")

plt.tight_layout()
plt.show()

In [ ]:
pd.Series({
    "R2 adj": model.rsquared_adj,
    "Shapiro-Wilk p-value": stats.shapiro(model.resid).pvalue,
    "Breusch-Pagan p-value": het_breuschpagan(model.resid, model.model.exog)[3],
}).to_frame("value")

Несмотря на хорошее описание средней зависимости, гетероскедастичность остается. Используем устойчивую к ней оценку ковариационной матрицы **HC3**.

HC3 корректирует стандартные ошибки, p-value и интервалы. Коэффициенты, предсказания и остатки остаются прежними; сама гетероскедастичность не исчезает.

In [ ]:
model_hc3 = ols("Volume ~ I(Girth ** 2):Height", trees).fit(
    cov_type="HC3", use_t=True,
)

pd.DataFrame({
    "coef": model.params,
    "SE OLS": model.bse,
    "SE HC3": model_hc3.bse,
    "p-value OLS": model.pvalues,
    "p-value HC3": model_hc3.pvalues,
}).join(model_hc3.conf_int().rename(columns={0: "CI low HC3", 1: "CI high HC3"}))

### Задание 4

Для **Chinstrap** постройте модель глубины клюва `bill_depth_mm`. Выберите не более трех исходных признаков; при необходимости добавьте одно обоснованное взаимодействие или преобразование.

Сравните простую и расширенную модели, обоснуйте итоговую формулу. Выведите коэффициенты с 95%-ми интервалами, скорректированный $R^2$ и RMSE. Проверьте остатки и гомоскедастичность, при необходимости используйте HC3 или преобразование переменных. Убедитесь, что результат не определяется одним необычным наблюдением. Модель должна оставаться компактной и устойчивой.

In [ ]:
# Your code here